In [64]:
import json, shutil
from pathlib import Path

In [65]:
LABEL_DIR = Path("../data/labeled/labels")

# 기존 labels 폴더 삭제 후 재생성
if LABEL_DIR.exists():
    shutil.rmtree(LABEL_DIR)

LABEL_DIR.mkdir(parents=True, exist_ok=True)

print("labels 폴더 초기화 완료")

labels 폴더 초기화 완료


In [66]:
# 경로 설정
JSON_DIR = Path("../data/labeled/annotations")
LABEL_DIR = Path("../data/labeled/labels")

LABEL_DIR.mkdir(parents=True, exist_ok=True)

In [67]:
# 전체 라벨 종류 확인

all_labels = set()

for json_path in JSON_DIR.glob("*.json"):

    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    for shape in data["shapes"]:
        all_labels.add(shape["label"])

print("라벨 종류:")
print(sorted(all_labels))

라벨 종류:
['eye', 'user_character', 'user_id']


In [68]:
# 클래스 매핑
CLASS_MAP = {
    "user_character" : 0,
    "user_id" : 1
}

print(CLASS_MAP)

{'user_character': 0, 'user_id': 1}


In [69]:
# YOLO 형식 변환 함수
def convert_to_yolo(x1, y1, x2, y2, image_width, image_height):
    
    x_center = ( (x1 + x2) / 2 ) / image_width
    y_center = ( (y1 + y2) / 2 ) / image_height
    width = abs(x2 - x1) / image_width
    height = abs(y2 - y1) / image_height

    return(x_center, y_center, width, height)

In [70]:
# JSON 1개 테스트
json_file = next(JSON_DIR.glob("*.json"))

print(json_file)

..\data\labeled\annotations\ScreenShot2024_0304_084801113.json


In [71]:
# JSON 내용 읽기
with open(json_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print(data.keys())

dict_keys(['version', 'flags', 'shapes', 'imagePath', 'imageData', 'imageHeight', 'imageWidth', 'text'])


In [72]:
# 객체 정보 확인
for shape in data["shapes"]:
    print(shape["label"])

user_character
user_id


In [76]:
# JSON -> YOLO TXT 변환
for json_path in JSON_DIR.glob("*.json"):

    with open(json_path, "r", encoding='utf-8') as f:
        data = json.load(f)

    
    image_width = data["imageWidth"]
    image_height = data["imageHeight"]

    yolo_lines = []

    for shape in data["shapes"]:

        label = shape["label"]

        # 정의되지 않은 라벨은 건너뛰기
        if label not in CLASS_MAP:
            print(f"제외된 라벨 : {label}", )
            continue
        class_id = CLASS_MAP[label]
        points = shape["points"]

        x1, y1 = points[0]
        x2, y2 = points[1]

        x_center, y_center, width, height = convert_to_yolo(x1, y1, x2, y2, 
                                                            image_width, image_height)
        
        line = (
            f"{class_id} "
            f"{x_center:.6f} "
            f"{y_center:.6f} "
            f"{width:.6f} "
            f"{height:.6f}"
        )

        yolo_lines.append(line)


    txt_path = (LABEL_DIR / f"{json_path.stem}.txt")

    with open(txt_path, "w", encoding='utf-8') as f:
        f.write("\n".join(yolo_lines))

print("변환 완료")

제외된 라벨 : eye
변환 완료


In [77]:
# 생성 결과 확인
txt_files = list(LABEL_DIR.glob("*.txt"))

print("생성된 txt 수 : ", len(txt_files))

생성된 txt 수 :  907


In [78]:
# txt 내용 확인

sample_txt = txt_files[0]

with open(sample_txt, "r", encoding="utf-8") as f:

    print(f.read())

0 0.371326 0.629436 0.108139 0.182664
1 0.403823 0.441993 0.041034 0.027052
